# Boop — KataGo-Style AlphaZero Training

Trains a **CNN ResNet policy+value network** with MCTS self-play,
incorporating key improvements from KataGo:

| Improvement | Vanilla AlphaZero | This notebook |
|---|---|---|
| Network | MLP (3×256) | **6-block ResNet + SE attention**, CNN on 6×6 board |
| Input | Flat 184-float obs | **9-channel (5 board + 4 hand)** 6×6 spatial tensor |
| Training data | 1× per move | **8× via all 8 board symmetries** |
| MCTS sims | Fixed | **Playout cap randomization** (15 fast / 100 full) |
| Leaf eval | 1 leaf / forward pass | **Batched-leaf MCTS** (16 leaves / forward pass via virtual loss) |
| Temperature | Constant | **Annealing** — 1.0 first 30 moves, 0.05 after |
| LR | Cosine from ep 1 | **Linear warmup** (50 eps) then cosine decay |
| Optimizer | Adam | **AdamW** (true weight decay) |
| Entropy | None | **Entropy bonus** (prevents early mode collapse) |

### Network architecture
```
Input (9, 6, 6)  →  Stem (Conv 3×3, BN, ReLU)
                 →  6 × ResBlock (Conv-BN-ReLU-Conv-BN + SE) + skip
                 →  Policy head: 1×1 conv (2 ch) → flatten → Linear(108)
                 →  Value head:  1×1 conv (1 ch) → flatten → Linear(128) → ReLU → Linear(1) → Tanh
```

### KataGo improvements in detail
- **SE attention**: per-channel gating learns which feature maps matter for each position
- **Symmetry augmentation**: 4 rotations × 2 reflections = 8× free training data
- **Playout cap randomization**: 75% fast (15 sim) games for diversity + 25% full (100 sim) games for quality
- **Temperature annealing per game**: exploratory early, near-greedy for endgame decisions
- **LR warmup**: avoids destabilising BatchNorm layers with large early gradients
- **Entropy bonus**: small regulariser that keeps the policy spread out early in training
- **Batched-leaf MCTS**: collects a wave of leaves with *virtual loss*, then evaluates them all in one NN forward pass instead of one at a time. This is KataGo's core speedup; the gain scales with hardware parallelism (modest on CPU, large on GPU).

In [ ]:
%pip install open_spiel -q

In [ ]:
# Boop board game -- inlined so Colab works without the custom package.
# Faithful rules: graduation conserves pieces (kittens BECOME cats), and a
# player whose pool is empty must graduate a kitten of their choice rather than
# being stuck (which previously caused spurious draws).

import numpy as np
from open_spiel.python.observation import IIGObserverForPublicInfoGame
import pyspiel

_NUM_PLAYERS = 2
_ROWS = 6
_COLS = 6
_NUM_CELLS = _ROWS * _COLS
_NUM_PIECE_TYPES = 2
_NUM_PIECES = 8                              # each player has exactly 8 pieces
_GRADUATE_OFFSET = _NUM_PIECE_TYPES * _NUM_CELLS  # 72: graduation actions start here
_NUM_ACTIONS = _GRADUATE_OFFSET + _NUM_CELLS      # 108 = 72 placement + 36 graduate
_MAX_KITTENS = 8
_MAX_CATS = 8                               # all 8 pieces can become cats
_MAX_GAME_LENGTH = 500

_EMPTY = 0
_P0_KITTEN = 1
_P0_CAT = 2
_P1_KITTEN = 3
_P1_CAT = 4

_KITTEN_VAL = [_P0_KITTEN, _P1_KITTEN]
_CAT_VAL = [_P0_CAT, _P1_CAT]
_PIECE_VALS = [[_P0_KITTEN, _P0_CAT], [_P1_KITTEN, _P1_CAT]]

_GAME_TYPE = pyspiel.GameType(
    short_name='python_boop',
    long_name='Python Boop',
    dynamics=pyspiel.GameType.Dynamics.SEQUENTIAL,
    chance_mode=pyspiel.GameType.ChanceMode.DETERMINISTIC,
    information=pyspiel.GameType.Information.PERFECT_INFORMATION,
    utility=pyspiel.GameType.Utility.ZERO_SUM,
    reward_model=pyspiel.GameType.RewardModel.TERMINAL,
    max_num_players=_NUM_PLAYERS,
    min_num_players=_NUM_PLAYERS,
    provides_information_state_string=True,
    provides_information_state_tensor=False,
    provides_observation_string=True,
    provides_observation_tensor=True,
    parameter_specification={})

_GAME_INFO = pyspiel.GameInfo(
    num_distinct_actions=_NUM_ACTIONS,
    max_chance_outcomes=0,
    num_players=_NUM_PLAYERS,
    min_utility=-1.0,
    max_utility=1.0,
    utility_sum=0.0,
    max_game_length=_MAX_GAME_LENGTH)


class BoopGame(pyspiel.Game):
  def __init__(self, params=None):
    super().__init__(_GAME_TYPE, _GAME_INFO, params or dict())

  def new_initial_state(self):
    return BoopState(self)

  def make_py_observer(self, iig_obs_type=None, params=None):
    if ((iig_obs_type is None) or
        (iig_obs_type.public_info and not iig_obs_type.perfect_recall)):
      return BoopObserver(params)
    return IIGObserverForPublicInfoGame(iig_obs_type, params)


class BoopState(pyspiel.State):
  def __init__(self, game):
    super().__init__(game)
    self._cur_player = 0
    self._is_terminal = False
    self._winner = None
    self._move_count = 0
    self.board = np.zeros((_ROWS, _COLS), dtype=np.int8)
    self._hand = [[_MAX_KITTENS, 0], [_MAX_KITTENS, 0]]

  def current_player(self):
    return pyspiel.PlayerId.TERMINAL if self._is_terminal else self._cur_player

  def _legal_actions(self, player):
    # Forced-graduation rule: if the pool is empty, the player must graduate one
    # of their kittens on the board (returning it to the pool as a cat) instead
    # of placing. Graduation actions are _GRADUATE_OFFSET + cell.
    if self._hand[player][0] == 0 and self._hand[player][1] == 0:
      kitten_val = _KITTEN_VAL[player]
      return [_GRADUATE_OFFSET + r * _COLS + c
              for r in range(_ROWS) for c in range(_COLS)
              if self.board[r, c] == kitten_val]
    actions = []
    for r in range(_ROWS):
      for c in range(_COLS):
        if self.board[r, c] == _EMPTY:
          cell = r * _COLS + c
          if self._hand[player][0] > 0:
            actions.append(cell)
          if self._hand[player][1] > 0:
            actions.append(_NUM_CELLS + cell)
    return sorted(actions)

  def _apply_action(self, action):
    p = self._cur_player
    if action >= _GRADUATE_OFFSET:
      # Forced graduation: a kitten on the board becomes a cat in the pool.
      cell = action - _GRADUATE_OFFSET
      r, c = cell // _COLS, cell % _COLS
      self.board[r, c] = _EMPTY
      self._hand[p][1] += 1
      self._move_count += 1
      self._post_move(p)        # no piece placed → no boop
      return
    piece_type = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    self._hand[p][piece_type] -= 1
    self.board[r, c] = _PIECE_VALS[p][piece_type]
    self._boop(r, c, is_cat=(piece_type == 1))
    self._move_count += 1
    self._post_move(p)

  def _post_move(self, p):
    """Shared post-move resolution: win checks, promotion, turn handoff."""
    if self._move_count >= _MAX_GAME_LENGTH:
      self._is_terminal = True
      return
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._promote_kittens(p)
    self._promote_kittens(1 - p)
    for player in (p, 1 - p):
      if self._check_win(player):
        self._is_terminal = True
        self._winner = player
        return
    self._cur_player = 1 - p
    # With forced graduation a player is never permanently stuck: an empty pool
    # forces a graduation, and a board of eight cats is already a win. The guard
    # below is defensive only and should not trigger in normal play.
    if not self._legal_actions(self._cur_player):
      self._is_terminal = True

  def _action_to_string(self, player, action):
    if action >= _GRADUATE_OFFSET:
      cell = action - _GRADUATE_OFFSET
      r, c = cell // _COLS, cell % _COLS
      return f'p{player}:graduate@({r},{c})'
    pt = action // _NUM_CELLS
    cell = action % _NUM_CELLS
    r, c = cell // _COLS, cell % _COLS
    piece = 'cat' if pt else 'kitten'
    return f'p{player}:{piece}@({r},{c})'

  def is_terminal(self):
    return self._is_terminal

  def returns(self):
    if self._winner == 0:
      return [1.0, -1.0]
    if self._winner == 1:
      return [-1.0, 1.0]
    return [0.0, 0.0]

  def __str__(self):
    syms = {
        _EMPTY: '.', _P0_KITTEN: 'k', _P0_CAT: 'K',
        _P1_KITTEN: 'o', _P1_CAT: 'O',
    }
    rows = [
        ''.join(syms[self.board[r, c]] for c in range(_COLS))
        for r in range(_ROWS)
    ]
    rows.append(
        f'P0: {self._hand[0][0]}k {self._hand[0][1]}K  '
        f'P1: {self._hand[1][0]}k {self._hand[1][1]}K  '
        f'move={self._move_count}')
    return '\n'.join(rows)

  def _boop(self, r, c, is_cat):
    for dr in (-1, 0, 1):
      for dc in (-1, 0, 1):
        if dr == 0 and dc == 0:
          continue
        nr, nc = r + dr, c + dc
        if not (0 <= nr < _ROWS and 0 <= nc < _COLS):
          continue
        neighbor = self.board[nr, nc]
        if neighbor == _EMPTY:
          continue
        neighbor_is_cat = neighbor in (_P0_CAT, _P1_CAT)
        if not is_cat and neighbor_is_cat:
          continue
        dest_r, dest_c = nr + dr, nc + dc
        owner = 0 if neighbor in (_P0_KITTEN, _P0_CAT) else 1
        n_type = 1 if neighbor_is_cat else 0
        if not (0 <= dest_r < _ROWS and 0 <= dest_c < _COLS):
          self.board[nr, nc] = _EMPTY
          self._hand[owner][n_type] += 1
        elif self.board[dest_r, dest_c] == _EMPTY:
          self.board[dest_r, dest_c] = neighbor
          self.board[nr, nc] = _EMPTY

  def _promote_kittens(self, player):
    kitten_val = _KITTEN_VAL[player]
    to_promote = set()
    for r in range(_ROWS):
      for c in range(_COLS):
        for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
          cells = []
          for k in range(3):
            nr, nc = r + dr * k, c + dc * k
            if (0 <= nr < _ROWS and 0 <= nc < _COLS
                and self.board[nr, nc] == kitten_val):
              cells.append((nr, nc))
            else:
              break
          if len(cells) == 3:
            to_promote.update(cells)
    if not to_promote:
      return
    # Faithful rule: the three(+) kittens BECOME cats. Remove them from the board
    # and add the same number of cats to the pool — total pieces are conserved.
    for pr, pc in to_promote:
      self.board[pr, pc] = _EMPTY
    self._hand[player][1] += len(to_promote)

  def _check_win(self, player):
    cat_val = _CAT_VAL[player]
    # Win condition 1: all eight pieces on the board as cats.
    if int(np.sum(self.board == cat_val)) >= _NUM_PIECES:
      return True
    # Win condition 2: three cats in a row (orthogonal or diagonal).
    for r in range(_ROWS):
      for c in range(_COLS):
        for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
          if all(
              0 <= r + dr * k < _ROWS
              and 0 <= c + dc * k < _COLS
              and self.board[r + dr * k, c + dc * k] == cat_val
              for k in range(3)):
            return True
    return False


class BoopObserver:
  def __init__(self, params):
    if params:
      raise ValueError(f'Observation parameters not supported; passed {params}')
    board_size = 5 * _ROWS * _COLS
    self.tensor = np.zeros(board_size + 4, np.float32)
    self.dict = {
        'observation': np.reshape(self.tensor[:board_size], (5, _ROWS, _COLS)),
        'hand': self.tensor[board_size:],
    }

  def set_from(self, state, player):
    self.tensor.fill(0)
    obs = self.dict['observation']
    hand = self.dict['hand']
    opp = 1 - player
    mk, mc = _KITTEN_VAL[player], _CAT_VAL[player]
    ok, oc = _KITTEN_VAL[opp], _CAT_VAL[opp]
    for r in range(_ROWS):
      for c in range(_COLS):
        v = state.board[r, c]
        if v == _EMPTY:   obs[0, r, c] = 1.0
        elif v == mk:     obs[1, r, c] = 1.0
        elif v == mc:     obs[2, r, c] = 1.0
        elif v == ok:     obs[3, r, c] = 1.0
        elif v == oc:     obs[4, r, c] = 1.0
    hand[0] = state._hand[player][0] / _MAX_KITTENS
    hand[1] = state._hand[player][1] / _MAX_CATS
    hand[2] = state._hand[opp][0] / _MAX_KITTENS
    hand[3] = state._hand[opp][1] / _MAX_CATS

  def string_from(self, state, player):
    del player
    return str(state)


pyspiel.register_game(_GAME_TYPE, BoopGame)


In [ ]:
game = pyspiel.load_game('python_boop')
state = game.new_initial_state()
print('Game:', game.get_type().long_name)
print('Actions:', game.num_distinct_actions())
print('Obs size:', game.observation_tensor_size())
print()
print(state)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import random

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


# ── Input helpers ──────────────────────────────────────────────────────────────────────────────

def _obs_to_9ch(obs_np):
    """Flat 184-float obs → (9, 6, 6): 5 board planes + 4 hand scalars broadcast."""
    board = obs_np[..., :180].reshape(*obs_np.shape[:-1], 5, 6, 6)
    hand  = obs_np[..., 180:]
    # broadcast hand scalars spatially so the CNN sees them at every cell
    hand_planes = np.broadcast_to(
        hand[..., None, None], hand.shape + (6, 6)).copy()
    return np.concatenate([board, hand_planes], axis=-3)   # (..., 9, 6, 6)


def state_to_tensor(state, device):
    """Single game state → (1, 9, 6, 6) float tensor."""
    obs = np.array(state.observation_tensor(state.current_player()), dtype=np.float32)
    x   = _obs_to_9ch(obs)[None]        # (1, 9, 6, 6)
    return torch.from_numpy(x).to(device)


def batch_to_tensor(obs_list, device):
    """List of flat 184-float observations → (B, 9, 6, 6) float tensor."""
    obs = np.stack(obs_list).astype(np.float32)   # (B, 184)
    x   = _obs_to_9ch(obs)                         # (B, 9, 6, 6)
    return torch.from_numpy(x).to(device)


# ── Network modules ────────────────────────────────────────────────────────────────────────────

class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention (KataGo-style)."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels * 2),
        )

    def forward(self, x):
        s = x.mean(dim=(2, 3))             # global avg pool → (B, C)
        scale, bias = self.fc(s).chunk(2, dim=1)
        scale = torch.sigmoid(scale)
        return (x * scale[:, :, None, None]
                  + bias[:, :, None, None])


class ResBlock(nn.Module):
    """Residual block: Conv-BN-ReLU-Conv-BN + SE attention + skip."""
    def __init__(self, channels, use_se=True):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.se  = SEBlock(channels) if use_se else nn.Identity()
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.se(self.net(x)) + x)


class BoopNet(nn.Module):
    """
    KataGo-style network for Boop.

    Input  : (B, 9, 6, 6) — 5 board planes + 4 hand scalars broadcast
    Body   : Conv stem → N × ResBlock(channels, SE)
    Policy : 1×1 conv (2 ch) → flatten → Linear(_NUM_ACTIONS=108)
    Value  : 1×1 conv (1 ch) → flatten → Linear(128) → ReLU → Linear(1) → Tanh
    """
    def __init__(self, channels=128, num_blocks=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(9, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
        self.body = nn.Sequential(*[ResBlock(channels) for _ in range(num_blocks)])

        self.policy_head = nn.Sequential(
            nn.Conv2d(channels, 2, 1, bias=False),
            nn.BatchNorm2d(2),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(2 * 6 * 6, _NUM_ACTIONS),
        )
        self.value_head = nn.Sequential(
            nn.Conv2d(channels, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(36, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        x = self.body(self.stem(x))
        return self.policy_head(x), self.value_head(x)


print(f'Device: {DEVICE}')
_demo = BoopNet()
print(f'BoopNet params: {sum(p.numel() for p in _demo.parameters()):,}')
del _demo

In [ ]:
from open_spiel.python.algorithms import mcts as mcts_lib


# ── 8-fold board symmetry augmentation ─────────────────────────────────────────
# The 6×6 Boop board has 8 dihedral symmetries (4 rotations × 2 reflections).
# Every self-play sample is augmented with all 8 copies for free training data.

def _compute_action_perm(sym_idx):
    """
    Precompute the 108-element action permutation for board symmetry sym_idx.
    Covers both placement actions (2 piece types x 36 cells) and the 36
    graduation actions, which transform spatially the same way as a cell.
    Symmetries 0-3: 0°/90°/180°/270° CCW rotation.
    Symmetries 4-7: horizontal flip followed by the same rotations.
    """
    k    = sym_idx % 4
    flip = sym_idx >= 4
    perm = np.empty(_NUM_ACTIONS, dtype=np.int32)
    for pt in range(2):
        for r in range(6):
            for c in range(6):
                nr, nc = r, c
                if flip:
                    nc = 5 - nc
                for _ in range(k):       # k × 90° CCW: (r,c) → (5-c, r)
                    nr, nc = 5 - nc, nr
                perm[pt * 36 + r * 6 + c] = pt * 36 + nr * 6 + nc
    for r in range(6):                   # graduation actions (cell-indexed)
        for c in range(6):
            nr, nc = r, c
            if flip:
                nc = 5 - nc
            for _ in range(k):
                nr, nc = 5 - nc, nr
            perm[_GRADUATE_OFFSET + r * 6 + c] = _GRADUATE_OFFSET + nr * 6 + nc
    return perm

_ACTION_PERMS = [_compute_action_perm(i) for i in range(8)]


def augment_sample(obs_flat, policy, value):
    """Return all 8 symmetry copies of a (obs, policy, value) training sample."""
    obs   = np.asarray(obs_flat, dtype=np.float32)
    board = obs[:180].reshape(5, 6, 6)
    hand  = obs[180:]                       # hand counts: spatially invariant
    pol   = np.asarray(policy, dtype=np.float32)

    out = []
    for sym_idx in range(8):
        k    = sym_idx % 4
        flip = sym_idx >= 4
        b    = board.copy()
        if flip:
            b = b[:, :, ::-1]             # horizontal flip (cols)
        b = np.rot90(b, k=k, axes=(1, 2)) # k × 90° CCW
        aug_obs = np.concatenate([b.reshape(-1).copy(), hand])
        out.append((aug_obs, pol[_ACTION_PERMS[sym_idx]], value))
    return out


# ── MCTS evaluator ─────────────────────────────────────────────────────────────

class NNEvaluator(mcts_lib.Evaluator):
    """
    Wraps BoopNet as an OpenSpiel MCTS evaluator.
    Per-search cache: evaluate() and prior() share the same forward pass when
    they hit the same state (a leaf that later gets expanded).  Cache is cleared
    before each mcts_search() to stay consistent with network weights.
    """
    def __init__(self, network, device=DEVICE):
        self.network = network
        self.device  = device
        self._cache  = {}

    def clear_cache(self):
        self._cache = {}

    def _forward(self, state):
        key = str(state)
        if key not in self._cache:
            with torch.no_grad():
                logits, value = self.network(state_to_tensor(state, self.device))
            self._cache[key] = (logits.squeeze().cpu().numpy(), value.item())
        return self._cache[key]

    def evaluate(self, state):
        _, v = self._forward(state)
        cur  = state.current_player()
        ret  = [-v, -v]
        ret[cur] = v
        return ret

    def prior(self, state):
        legal = state.legal_actions()
        if not legal:
            return []
        logits, _ = self._forward(state)
        lg    = logits[legal] - logits[legal].max()
        probs = np.exp(lg)
        probs /= probs.sum()
        return list(zip(legal, probs.tolist()))


def make_bot(game, evaluator, num_simulations):
    return mcts_lib.MCTSBot(
        game,
        uct_c=1.4,
        max_simulations=num_simulations,
        evaluator=evaluator,
        dirichlet_noise=(0.25, 0.3),
    )


# ── Self-play ──────────────────────────────────────────────────────────────────

def self_play_game(game, bot, temp_threshold=30, evaluator=None):
    """
    MCTS self-play with temperature annealing:
      moves < temp_threshold : temperature = 1.0  (explore)
      moves >= temp_threshold: temperature = 0.05 (near-greedy for endgame)
    Returns list of (obs_flat, policy_vec, game_return) per move.
    """
    state    = game.new_initial_state()
    history  = []
    move_num = 0

    while not state.is_terminal():
        player      = state.current_player()
        obs         = state.observation_tensor(player)
        temperature = 1.0 if move_num < temp_threshold else 0.05

        # Flush per-search cache so evaluate() and prior() share computation
        # for the same leaf state within this search, without cross-search staleness.
        if evaluator is not None:
            evaluator.clear_cache()
        root      = bot.mcts_search(state)
        legal     = state.legal_actions()
        visit_map = {ch.action: ch.explore_count for ch in root.children}
        counts    = np.array([visit_map.get(a, 0) for a in legal], dtype=np.float32)

        if temperature < 1e-2 or counts.sum() == 0:
            probs          = np.zeros(len(legal), dtype=np.float32)
            probs[counts.argmax()] = 1.0
        else:
            log_ct = np.log(counts + 1e-10) / temperature
            log_ct -= log_ct.max()
            ct     = np.exp(log_ct)
            probs  = ct / ct.sum()

        policy_vec = np.zeros(_NUM_ACTIONS, dtype=np.float32)
        for a, p in zip(legal, probs):
            policy_vec[a] = p

        action = np.random.choice(legal, p=probs)
        history.append((player, list(obs), policy_vec))
        state.apply_action(action)
        move_num += 1

    returns = state.returns()
    return [(obs, pol, returns[pl]) for pl, obs, pol in history]


# ── Evaluation ─────────────────────────────────────────────────────────────────

def greedy_action(network, state, device=DEVICE, temperature=0.0):
    with torch.no_grad():
        logits, _ = network(state_to_tensor(state, device))
    logits = logits.squeeze().cpu().numpy()
    legal  = state.legal_actions()
    if temperature == 0.0:
        return legal[int(logits[legal].argmax())]
    lg = logits[legal] - logits[legal].max()
    probs = np.exp(lg / temperature)
    probs /= probs.sum()
    return int(np.random.choice(legal, p=probs))


def eval_vs_random(game, network, num_games=100, device=DEVICE):
    """Returns (wins, draws, losses) for network (greedy, P0) vs random (P1)."""
    wins, draws, losses = 0, 0, 0
    for _ in range(num_games):
        state = game.new_initial_state()
        while not state.is_terminal():
            if state.current_player() == 0:
                action = greedy_action(network, state, device)
            else:
                action = random.choice(state.legal_actions())
            state.apply_action(action)
        r = state.returns()[0]
        if r > 0:    wins   += 1
        elif r < 0:  losses += 1
        else:        draws  += 1
    return wins, draws, losses


def eval_vs_snapshot(game, network, snapshot, num_games=100, device=DEVICE):
    """Returns (wins, draws, losses); alternates colors and samples with temperature=1."""
    wins, draws, losses = 0, 0, 0
    for i in range(num_games):
        state = game.new_initial_state()
        net_player = i % 2  # alternate: net plays P0 in even games, P1 in odd
        while not state.is_terminal():
            net    = network if state.current_player() == net_player else snapshot
            action = greedy_action(net, state, device, temperature=1.0)
            state.apply_action(action)
        r = state.returns()[net_player]
        if r > 0:    wins   += 1
        elif r < 0:  losses += 1
        else:        draws  += 1
    return wins, draws, losses


def update_elo(elo, opp_elo, win_rate, k=32):
    expected = 1.0 / (1.0 + 10 ** ((opp_elo - elo) / 400.0))
    return elo + k * (win_rate - expected)


# ── Benchmark-pool evaluation: running-Elo tournaments (no fixed anchor) ─────────
# Every net (random, the untrained '0' net, and each checkpoint) is a tournament
# player entering at START_ELO and updated incrementally with the Elo K-factor.
# Evaluation is run in independent TRACKS: policy-only (0 sims) and one per MCTS
# budget. Each track keeps its own Elo table and its own head-to-head matrix, so
# `mcts50` ratings never mix with `policy` ratings.

import math
import itertools
from collections import defaultdict


def play_match(net_a, net_b, game, n_games, device, temp=1.0):
    """net_a vs net_b over n_games, alternating colours. net == None ==> random.
    Policy-only (no MCTS). Returns (wins_a, draws, wins_b) from net_a's view.
    Used by the quick eval as a lightweight progress pulse (does not touch Elo)."""
    wa = dd = wb = 0
    for i in range(n_games):
        state = game.new_initial_state()
        a_player = i % 2
        while not state.is_terminal():
            net = net_a if state.current_player() == a_player else net_b
            if net is None:
                action = random.choice(state.legal_actions())
            else:
                action = greedy_action(net, state, device, temperature=temp)
            state.apply_action(action)
        r = state.returns()[a_player]
        if r > 0:   wa += 1
        elif r < 0: wb += 1
        else:       dd += 1
    return wa, dd, wb


def _mcts_move(bot, state):
    """Most-visited action from an MCTS search (root Dirichlet noise gives the
    per-game variety that keeps a round-robin from replaying identical games)."""
    root   = bot.mcts_search(state)
    legal  = state.legal_actions()
    visits = {c.action: c.explore_count for c in root.children}
    counts = np.array([visits.get(a, 0) for a in legal], dtype=np.float32)
    return legal[int(counts.argmax())]


def run_tournament(players, elos, wdl, game, device,
                   games_per_pair=10, k=32.0, temp=1.0, sims=0):
    """One full round-robin for a single track. Moves are chosen by policy-only
    (sims == 0: greedy sampled at `temp`) or by MCTS with `sims` simulations
    (most-visited move). Every unordered pair plays `games_per_pair` games, half
    per colour, all shuffled into random order; Elo is updated after each game.
    `elos` must already hold a rating for every label. Mutates `elos` and `wdl`."""
    net  = {p['label']: p['net'] for p in players}
    bots = {}
    if sims > 0:
        bs = max(1, min(sims, 16))
        for p in players:
            if p['net'] is not None:
                bots[p['label']] = make_batched_bot(game, p['net'], device, sims, bs)

    def pick(label, state):
        if net[label] is None:
            return random.choice(state.legal_actions())
        if sims <= 0:
            return greedy_action(net[label], state, device, temperature=temp)
        return _mcts_move(bots[label], state)

    schedule = []
    for a, b in itertools.combinations(net.keys(), 2):
        half = games_per_pair // 2
        schedule += [(a, b)] * half
        schedule += [(b, a)] * half
        schedule += [(a, b)] * (games_per_pair - 2 * half)   # odd remainder
    random.shuffle(schedule)

    for p0, p1 in schedule:
        state = game.new_initial_state()
        while not state.is_terminal():
            lab = p0 if state.current_player() == 0 else p1
            state.apply_action(pick(lab, state))
        r  = state.returns()[0]
        s0 = 1.0 if r > 0 else (0.0 if r < 0 else 0.5)
        e0 = 1.0 / (1.0 + 10 ** ((elos[p1] - elos[p0]) / 400.0))
        elos[p0] += k * (s0 - e0)
        elos[p1] += k * ((1.0 - s0) - (1.0 - e0))
        cell = wdl[(p0, p1)]
        if r > 0:   cell[0] += 1
        elif r < 0: cell[2] += 1
        else:       cell[1] += 1


In [ ]:
import math


# ── Batched-leaf MCTS (KataGo's core speedup) ───────────────────────────────────
# Vanilla MCTS evaluates one leaf per NN forward pass. On CPU/GPU that wastes
# almost all throughput: a forward pass on a batch of 16 states costs barely more
# than a batch of 1. This bot collects a *wave* of `batch_size` leaves using
# **virtual loss**, evaluates them all in ONE forward pass, then backs them all
# up — typically 10-30× more simulations per second.
#
# Virtual loss: while a leaf is "in flight" (selected but not yet evaluated) we
# temporarily pretend the path lost. This pushes subsequent selections in the
# same wave down *different* branches so we gather a diverse batch instead of the
# same leaf 16 times. The virtual loss is removed when the real value is backed up.

class _BNode:
    """Lightweight search node with a virtual-loss counter."""
    __slots__ = ['action', 'player', 'prior', 'explore_count',
                 'total_reward', 'vloss', 'outcome', 'children']

    def __init__(self, action, player, prior):
        self.action        = action
        self.player        = player
        self.prior         = prior
        self.explore_count = 0
        self.total_reward  = 0.0
        self.vloss         = 0          # in-flight virtual visits
        self.outcome       = None
        self.children      = []


class BatchedMCTSBot:
    """
    Drop-in replacement for mcts_lib.MCTSBot exposing mcts_search(state) -> root.
    `root.children` are _BNode with .action and .explore_count, matching the
    interface self_play_game() consumes.

    Weights are read live from `network`, so in-place updates take effect
    immediately (no per-search cache needed — the batch IS the speedup).
    """
    def __init__(self, game, network, device, max_simulations,
                 batch_size=16, uct_c=1.4, dirichlet=(0.25, 0.3),
                 solve=True, random_state=None):
        self.game            = game
        self.network         = network
        self.device          = device
        self.max_simulations = max_simulations
        self.batch_size      = batch_size
        self.uct_c           = uct_c
        self.dirichlet       = dirichlet
        self.solve           = solve
        self.max_utility     = game.max_utility()
        self._rng            = random_state or np.random.RandomState()

    # ── NN: evaluate a list of states in one forward pass ──────────────────────
    def _eval_batch(self, states):
        obs_list = [s.observation_tensor(s.current_player()) for s in states]
        x = batch_to_tensor(obs_list, self.device)          # (B, 9, 6, 6)
        with torch.no_grad():
            logits, values = self.network(x)
        return (logits.cpu().numpy(),                       # (B, _NUM_ACTIONS)
                values.squeeze(-1).cpu().numpy())           # (B,)

    # ── Expand a node: turn NN policy logits into prior-weighted children ──────
    def _expand(self, node, cur_player, legal, logits, add_noise):
        lg = logits[legal] - logits[legal].max()
        pr = np.exp(lg)
        pr /= pr.sum()
        if add_noise and self.dirichlet is not None:
            eps, alpha = self.dirichlet
            noise = self._rng.dirichlet([alpha] * len(legal))
            pr = (1.0 - eps) * pr + eps * noise
        children = [_BNode(a, cur_player, float(p)) for a, p in zip(legal, pr)]
        self._rng.shuffle(children)                         # reduce move-order bias
        node.children = children

    # ── PUCT with virtual loss folded in ───────────────────────────────────────
    def _puct(self, child, parent_n_eff):
        if child.outcome is not None:
            return child.outcome[child.player]
        ec = child.explore_count + child.vloss             # effective visits
        q  = (child.total_reward - child.vloss) / ec if ec > 0 else 0.0
        return q + self.uct_c * child.prior * math.sqrt(parent_n_eff) / (ec + 1)

    # ── Descend from root to an unexpanded/terminal leaf, applying virtual loss ─
    def _select_leaf(self, root, root_state):
        path  = [root]
        root.vloss += 1
        state = root_state.clone()
        node  = root
        while node.children and not state.is_terminal():
            parent_n_eff = node.explore_count + node.vloss
            best = max(node.children, key=lambda c: self._puct(c, parent_n_eff))
            state.apply_action(best.action)
            best.vloss += 1
            path.append(best)
            node = best
        return path, state, node

    # ── Back up a network value estimate (non-terminal leaf) ───────────────────
    def _backup_value(self, path, leaf_cur, value):
        for node in reversed(path):
            node.vloss         -= 1
            node.explore_count += 1
            node.total_reward  += value if node.player == leaf_cur else -value

    # ── Back up a terminal result, propagating solved outcomes (MCTS-Solver) ───
    def _backup_terminal(self, path, returns):
        path[-1].outcome = returns
        solved = self.solve
        for node in reversed(path):
            node.vloss         -= 1
            node.explore_count += 1
            node.total_reward  += returns[node.player]
            if solved and node.children:
                player = node.children[0].player
                best, all_solved = None, True
                for ch in node.children:
                    if ch.outcome is None:
                        all_solved = False
                    elif best is None or ch.outcome[player] > best.outcome[player]:
                        best = ch
                if best is not None and (all_solved or
                                         best.outcome[player] == self.max_utility):
                    node.outcome = best.outcome
                else:
                    solved = False

    # ── Main search: waves of batched-leaf evaluation ──────────────────────────
    def mcts_search(self, state):
        root = _BNode(None, state.current_player(), 1.0)

        # Seed: expand root (with Dirichlet noise) and back up its own value.
        logits, values = self._eval_batch([state])
        self._expand(root, state.current_player(), state.legal_actions(),
                     logits[0], add_noise=True)
        root.explore_count = 1
        root.total_reward += float(values[0])

        while root.explore_count < self.max_simulations:
            if root.outcome is not None:
                break
            wave = min(self.batch_size, self.max_simulations - root.explore_count)

            # 1. Collect a diverse wave of leaves via virtual loss.
            pending = []
            for _ in range(wave):
                if root.outcome is not None:
                    break
                path, st, leaf = self._select_leaf(root, state)
                pending.append((path, leaf, st))

            # 2. Evaluate every unique non-terminal leaf in ONE forward pass.
            to_eval = {}
            for path, leaf, st in pending:
                if not st.is_terminal():
                    to_eval[id(leaf)] = (leaf, st)

            results = {}
            if to_eval:
                states = [s for (_, s) in to_eval.values()]
                lg, vv = self._eval_batch(states)
                for (k, (leaf, st)), logit_row, val in zip(to_eval.items(), lg, vv):
                    cur = st.current_player()
                    self._expand(leaf, cur, st.legal_actions(), logit_row, False)
                    results[k] = (cur, float(val))

            # 3. Back up all leaves (terminals resolved exactly; others via NN).
            for path, leaf, st in pending:
                if st.is_terminal():
                    self._backup_terminal(path, st.returns())
                else:
                    cur, val = results[id(leaf)]
                    self._backup_value(path, cur, val)

        return root


def make_batched_bot(game, network, device, num_simulations,
                     batch_size=16, uct_c=1.4, dirichlet=(0.25, 0.3)):
    return BatchedMCTSBot(game, network, device, num_simulations,
                          batch_size=batch_size, uct_c=uct_c, dirichlet=dirichlet)


In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# Colab GPU estimate: ~1-2 h for 5000 episodes with these settings.
NUM_EPISODES     = 5_000
FULL_MCTS_SIMS   = 100     # high-quality self-play (25% of games)
FAST_MCTS_SIMS   = 20      # fast self-play for diverse data (75% of games)
FAST_GAME_PROB   = 0.75    # fraction of episodes using FAST_MCTS_SIMS
MCTS_BATCH_SIZE  = 16      # leaves evaluated per NN forward pass (batched MCTS)
TEMP_THRESHOLD   = 30      # move number to switch from temp=1.0 to temp=0.05
EVAL_EVERY       = 50      # QUICK eval: live net vs the most recent benchmark
DEEP_EVAL_EVERY  = 500     # DEEP eval: full round-robin tournaments + new benchmark
QUICK_EVAL_GAMES = 20      # games vs the most recent benchmark (quick pulse)
TOURNEY_GAMES_PER_PAIR = 10  # round-robin games per pair (5 each colour)
# Independent evaluation tracks: (name, MCTS simulations per move). Each track
# keeps its OWN Elo table and W/D/L matrix. sims=0 is the raw policy network.
EVAL_TRACKS      = [('policy', 0), ('mcts25', 25), ('mcts50', 50)]
EVAL_TEMP        = 1.0     # sampling temperature for policy-track eval moves
START_ELO        = 1000.0  # random and the initial net seed here
K_FACTOR         = 32.0    # Elo update step
BATCH_SIZE       = 512
TRAIN_STEPS_PER_EP = 8     # gradient steps per self-play game
MAX_BUFFER       = 200_000  # 8× augmentation inflates buffer; cap it
CHANNELS         = 64
NUM_BLOCKS       = 4
LR_PEAK          = 2e-3
LR_WARMUP_EPS    = 100      # linear warmup episodes
WEIGHT_DECAY     = 1e-4
ENTROPY_BONUS    = 0.01     # policy entropy regularisation coefficient

# ── Setup ─────────────────────────────────────────────────────────────────────
game         = pyspiel.load_game('python_boop')
base_network = BoopNet(CHANNELS, NUM_BLOCKS).to(DEVICE)   # uncompiled; shares
network      = base_network                               # weights with compiled

# torch.compile() fuses ops and JIT-compiles the network (dynamic=True → one
# shape-agnostic graph, since MCTS feeds variable batch sizes). It helps on GPU
# but needs a working C++ backend, which native Windows often lacks (Inductor
# raises "Compiler: cl is not found."). torch.compile fails lazily on the FIRST
# forward, so we trigger a dummy forward here and fall back to eager mode if the
# backend can't build. Snapshots deepcopy the uncompiled `base_network` anyway.
# (The batched-leaf MCTS is the main speedup, not compile.)
network = base_network
if hasattr(torch, 'compile'):
    try:
        _compiled = torch.compile(base_network, dynamic=True)
        with torch.no_grad():                       # force compilation now
            _compiled(torch.zeros(1, 9, 6, 6, device=DEVICE))
        network = _compiled
        print('torch.compile: enabled')
    except Exception as _e:
        print(f'torch.compile: disabled ({type(_e).__name__}) — running eager mode')

optimizer = torch.optim.AdamW(network.parameters(),
                               lr=LR_PEAK, weight_decay=WEIGHT_DECAY)

def _lr_lambda(ep):
    if ep < LR_WARMUP_EPS:
        return ep / max(LR_WARMUP_EPS, 1)
    frac = (ep - LR_WARMUP_EPS) / max(NUM_EPISODES - LR_WARMUP_EPS, 1)
    return 0.5 * (1.0 + np.cos(np.pi * frac))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

# Batched-leaf MCTS: both bots read weights live from `network` (in-place
# updates take effect immediately). Each search evaluates up to
# MCTS_BATCH_SIZE leaves per NN forward pass — KataGo's core speedup.
fast_bot  = make_batched_bot(game, network, DEVICE, FAST_MCTS_SIMS, MCTS_BATCH_SIZE)
full_bot  = make_batched_bot(game, network, DEVICE, FULL_MCTS_SIMS, MCTS_BATCH_SIZE)

replay_buffer = []
# Tournament pool. `random` and the untrained `0` net both start at START_ELO.
_init_snap = copy.deepcopy(base_network).to(DEVICE); _init_snap.eval()
benchmarks = [{'label': 'random', 'net': None},
              {'label': '0',      'net': _init_snap}]
# Per-track Elo tables and head-to-head matrices (independent between tracks).
elos = {name: {'random': START_ELO, '0': START_ELO} for name, _ in EVAL_TRACKS}
wdl  = {name: defaultdict(lambda: [0, 0, 0])         for name, _ in EVAL_TRACKS}
hist = {'ep': [], 'p_loss': [], 'v_loss': [],
        'quick_ep': [], 'q_w': [], 'q_d': [], 'q_l': [],
        'deep_ep': [], 'elo_snap': []}   # elo_snap: list of {track: {label: elo}}


def _run_all_tracks():
    for name, sims in EVAL_TRACKS:
        run_tournament(benchmarks, elos[name], wdl[name], game, DEVICE,
                       TOURNEY_GAMES_PER_PAIR, K_FACTOR, EVAL_TEMP, sims)
    hist['deep_ep'].append(ep_now[0])
    hist['elo_snap'].append({name: dict(elos[name]) for name, _ in EVAL_TRACKS})


def _print_tracks(prefix):
    for name, _ in EVAL_TRACKS:
        ladder = '  '.join(f'{b["label"]}={elos[name][b["label"]]:.0f}'
                           for b in sorted(benchmarks, key=lambda b: -elos[name][b['label']]))
        print(f'{prefix} {name:>7} Elos (strong→weak): {ladder}')


n_params = sum(p.numel() for p in network.parameters() if p.requires_grad)
print(f'Device: {DEVICE}  |  BoopNet params: {n_params:,}')
print(f'{NUM_EPISODES} eps | fast {FAST_MCTS_SIMS} sims ({FAST_GAME_PROB:.0%}) '
      f'/ full {FULL_MCTS_SIMS} sims | batch {MCTS_BATCH_SIZE} | 8× aug | {TRAIN_STEPS_PER_EP} steps/ep')
print(f'Quick eval every {EVAL_EVERY} eps (vs latest benchmark); deep tournaments '
      f'every {DEEP_EVAL_EVERY} eps | tracks: {[f"{n}({s})" for n, s in EVAL_TRACKS]}')
print('-' * 72)

# Iteration-0 tournaments: random vs the untrained net, one per track.
ep_now = [0]
_run_all_tracks()
_print_tracks('Iter     0 |')
print('-' * 72)

for ep in range(1, NUM_EPISODES + 1):
    ep_now[0] = ep

    # 1. Self-play with playout cap randomization
    network.eval()
    bot      = fast_bot if random.random() < FAST_GAME_PROB else full_bot
    raw_data = self_play_game(game, bot, temp_threshold=TEMP_THRESHOLD)

    # 2. Augment each move with all 8 board symmetries (8× free data)
    for obs, pol, val in raw_data:
        replay_buffer.extend(augment_sample(obs, pol, val))
    if len(replay_buffer) > MAX_BUFFER:
        del replay_buffer[:-MAX_BUFFER]

    # 3. Train
    network.train()
    p_losses, v_losses = [], []
    if len(replay_buffer) >= BATCH_SIZE:
        for _ in range(TRAIN_STEPS_PER_EP):
            batch  = random.sample(replay_buffer, BATCH_SIZE)
            x_b    = batch_to_tensor([s[0] for s in batch], DEVICE)
            pol_b  = torch.from_numpy(
                         np.stack([s[1] for s in batch]).astype(np.float32)
                     ).to(DEVICE)
            val_b  = torch.from_numpy(
                         np.array([[s[2]] for s in batch], dtype=np.float32)
                     ).to(DEVICE)

            logits, value = network(x_b)
            log_p  = F.log_softmax(logits, dim=1)

            p_loss = -(pol_b * log_p).sum(dim=1).mean()
            entropy = -(torch.exp(log_p) * log_p).sum(dim=1).mean()
            v_loss = F.mse_loss(value, val_b)
            loss = p_loss - ENTROPY_BONUS * entropy + v_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            p_losses.append(p_loss.item())
            v_losses.append(v_loss.item())
        scheduler.step()

    if ep % EVAL_EVERY != 0:
        continue

    # 4. Evaluation
    network.eval()
    mp = float(np.mean(p_losses)) if p_losses else float('nan')
    mv = float(np.mean(v_losses)) if v_losses else float('nan')
    hist['ep'].append(ep); hist['p_loss'].append(mp); hist['v_loss'].append(mv)
    lr_now = optimizer.param_groups[0]['lr']

    if ep % DEEP_EVAL_EVERY == 0:
        # DEEP: add current net as a new benchmark, warm-starting each track's Elo
        # from the previous checkpoint, then run one round-robin per track.
        new_label = str(ep)
        for name, _ in EVAL_TRACKS:
            elos[name][new_label] = elos[name][benchmarks[-1]['label']]
        snap = copy.deepcopy(base_network).to(DEVICE); snap.eval()
        benchmarks.append({'label': new_label, 'net': snap})
        _run_all_tracks()
        print(f'Ep {ep:5d} | p={mp:.3f} v={mv:.3f} | DEEP tournaments '
              f'(pool {len(benchmarks)}) | lr={lr_now:.2e}')
        _print_tracks('        |')
    else:
        # QUICK: policy-only pulse vs the most recent benchmark (no Elo change).
        ref = benchmarks[-1]
        w, d, l = play_match(network, ref['net'], game,
                             QUICK_EVAL_GAMES, DEVICE, temp=EVAL_TEMP)
        hist['quick_ep'].append(ep)
        hist['q_w'].append(w); hist['q_d'].append(d); hist['q_l'].append(l)
        print(f'Ep {ep:5d} | p={mp:.3f} v={mv:.3f} | vs {ref["label"]:>5} '
              f'W{w:2d}D{d:2d}L{l:2d} (policy Elo={elos["policy"][ref["label"]]:.0f}) '
              f'| lr={lr_now:.2e}')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle('Boop KataGo-Style Training', fontsize=14)

axes[0, 0].plot(hist['ep'], hist['p_loss'])
axes[0, 0].set_title('Policy Loss (cross-entropy)')
axes[0, 0].set_xlabel('Episode')

axes[0, 1].plot(hist['ep'], hist['v_loss'], color='tab:orange')
axes[0, 1].set_title('Value Loss (MSE)')
axes[0, 1].set_xlabel('Episode')

# Elo of the model trained-to-episode-N, one line per evaluation track.
# (At each deep eval the newest benchmark's rating under each search budget.)
def _newest_label(ep):
    return '0' if ep == 0 else str(ep)
for name, _ in EVAL_TRACKS:
    xs, ys = [], []
    for ep, snap in zip(hist['deep_ep'], hist['elo_snap']):
        lbl = _newest_label(ep)
        if lbl in snap[name]:
            xs.append(ep); ys.append(snap[name][lbl])
    axes[0, 2].plot(xs, ys, marker='.', label=name)
axes[0, 2].axhline(START_ELO, color='gray', linestyle='--', linewidth=0.8)
axes[0, 2].set_title('Current-model Elo by track')
axes[0, 2].set_xlabel('Episode'); axes[0, 2].legend(fontsize=8)

# Quick-eval progress pulse: policy-only W/D/L vs the most recent benchmark.
axes[1, 0].plot(hist['quick_ep'], hist['q_w'], color='tab:green', marker='.', label='Win')
axes[1, 0].plot(hist['quick_ep'], hist['q_d'], color='gray', linestyle='--', label='Draw')
axes[1, 0].plot(hist['quick_ep'], hist['q_l'], color='tab:red', linestyle=':', label='Loss')
axes[1, 0].set_title(f'Quick eval vs latest benchmark ({QUICK_EVAL_GAMES} games, policy)')
axes[1, 0].set_xlabel('Episode'); axes[1, 0].legend(fontsize=8)

# Final Elo per benchmark, grouped bars by track.
labels = [b['label'] for b in benchmarks]
x = np.arange(len(labels)); width = 0.8 / max(len(EVAL_TRACKS), 1)
for k, (name, _) in enumerate(EVAL_TRACKS):
    vals = [elos[name].get(l, np.nan) for l in labels]
    axes[1, 1].bar(x + (k - (len(EVAL_TRACKS) - 1) / 2) * width, vals, width, label=name)
axes[1, 1].axhline(START_ELO, color='gray', linestyle='--', linewidth=0.8)
axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1, 1].set_title('Final Elo per benchmark'); axes[1, 1].set_ylabel('Elo')
axes[1, 1].legend(fontsize=8)

# Head-to-head win-rate matrix for the strongest track (highest sims).
mtrack = EVAL_TRACKS[-1][0]
W = wdl[mtrack]
n = len(labels)
M = np.full((n, n), np.nan)
for i, ri in enumerate(labels):
    for j, cj in enumerate(labels):
        if (ri, cj) in W:
            w, d, l = W[(ri, cj)]; tot = w + d + l
            if tot: M[i, j] = (w + 0.5 * d) / tot
        elif (cj, ri) in W:
            w, d, l = W[(cj, ri)]; tot = w + d + l
            if tot: M[i, j] = 1.0 - (w + 0.5 * d) / tot
im = axes[1, 2].imshow(M, cmap='RdYlGn', vmin=0, vmax=1)
axes[1, 2].set_xticks(range(n)); axes[1, 2].set_yticks(range(n))
axes[1, 2].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1, 2].set_yticklabels(labels, fontsize=7)
axes[1, 2].set_title(f'Win-rate matrix — {mtrack} (row vs col)')
for i in range(n):
    for j in range(n):
        if not np.isnan(M[i, j]):
            axes[1, 2].text(j, i, f'{M[i, j]:.2f}', ha='center', va='center',
                            fontsize=6, color='black')
fig.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()
